# 第 5 章 Isaac 在机器人仿真中的应用


## 5.1.3 Isaac 的下载与安装


Isaac Sim 的安装方式主要有以下几种：



1. **通过 Omniverse Launcher 安装**（推荐）：下载并安装 NVIDIA Omniverse Launcher，在 Launcher 的 "Exchange" 页面中找到 Isaac Sim，一键下载安装。安装完成后可以直接从 Launcher 启动。

2. **通过源码安装 Isaac Lab**：Isaac Lab 是开源框架，可以在已安装 Isaac Sim 的基础上克隆并安装：



In [ ]:
# 克隆 Isaac Lab 仓库
git clone https://github.com/isaac-sim/IsaacLab.git
cd IsaacLab

# 创建并激活 conda 环境（需要 NVIDIA GPU 与 CUDA）
conda env create -f environment.yaml
conda activate isaaclab

# 安装 Isaac Lab 相关包
./isaaclab.sh --install


1. **通过 Docker 部署**：NVIDIA 提供 Isaac Sim 的 Docker 镜像，适合云端与服务器部署：



In [ ]:
docker pull nvcr.io/nvidia/isaac-sim:latest


需要说明的是，Isaac 依赖 NVIDIA GPU（RTX 系列）、CUDA 与 Windows/Linux 系统，安装包体积较大（数十 GB 级），建议在配置足够的机器上安装使用。安装完成后，可以在命令行中运行一个最小的 Python 脚本验证安装：



In [ ]:
from omni.isaac.kit import SimulationApp

# 以无头模式启动 Isaac Sim 应用
simulation_app = SimulationApp({"headless": True})
print("Isaac Sim 启动成功")
simulation_app.close()


在 Isaac 中建模与仿真时，通常遵循以下约定：

* **坐标系**：使用 USD 的右手坐标系，Z 轴向上（与 ROS 一致），单位通常为米（meter）。

* **关节约定**：关节角度使用弧度（radian）；通过 `ArticulationView` 的 `set_joint_positions` 等接口设置目标。

* **命名约定**：场景中的对象使用 `/World/xxx` 的绝对路径命名；机器人各连杆、关节名与 URDF/USD 中保持一致。

* **物理参数**：质量、惯性、摩擦、阻尼等参数在 USD 的物理属性中设置，或在加载 URDF/USD 后通过代码修改。



## 5.2.4 Isaac 中机器人概述


在 Isaac 中加载机器人模型有两种常用方式：加载 USD 文件，或加载 URDF（自动转换为 USD）。加载后，可以获取 Articulation 并控制关节。示例（以 Franka 机械臂为例）：



In [ ]:
from omni.isaac.core import World
from omni.isaac.core.utils.stage import add_reference_to_stage
from omni.isaac.core.articulations import ArticulationView

# 创建仿真世界
world = World(stage_units_in_meters=1.0)

# 将机器人 USD 添加到场景
usd_path = "/path/to/franka.usd"          # 替换为实际 USD 路径
add_reference_to_stage(usd_path, "/World/Franka")

world.reset()

# 获取 ArticulationView 并读取关节信息
franka = ArticulationView(prim_paths_expr="/World/Franka", name="franka_view")
joint_names = franka.dof_names
print("关节名称：", joint_names)


运行结果（以 Franka Panda 为例）为：



In [ ]:
关节名称： ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7', 'panda_finger_joint1', 'panda_finger_joint2']


## 5.3.1 开始使用 Isaac


使用 Isaac 编写仿真程序的基本流程是：启动 SimulationApp → 创建 World → 加载场景 / 机器人 → 进入仿真循环（步进 + 读取状态）。下面是一个完整的示例：



In [ ]:
from omni.isaac.kit import SimulationApp

# 1. 启动 Isaac Sim（headless=True 表示无窗口模式）
simulation_app = SimulationApp({"headless": True})

from omni.isaac.core import World
from omni.isaac.core.utils.stage import add_reference_to_stage

# 2. 创建仿真世界
world = World()

# 3. 加载机器人（以 USD 为例，路径需替换为实际文件）
add_reference_to_stage("/path/to/robot.usd", "/World/Robot")

# 4. 仿真主循环：步进 100 帧
for i in range(100):
    world.step(render=False)
    if i % 10 == 0:
        print("仿真帧：", i)

# 5. 关闭应用
simulation_app.close()


# 启动 Isaac Sim 图形界面./isaac-sim.sh# 以无头模式运行 Python 脚本（服务器/训练场景）./python.sh my_script.py# 运行 Isaac Lab 的测试/训练./isaaclab.sh --test./isaaclab.sh --train# 查看命令行帮助./isaac-sim.sh --help


> **说明：**
>
> 上述代码需要在安装 Isaac Sim/Lab 的环境中运行，并需准备对应的 USD 模型文件。Isaac 官方提供大量示例（
>
> `/isaac-sim/standalone_examples/`
>
> ）可供参考。



## 5.3.3 写 Isaac 文档


Isaac 的扩展都带有文档字符串与示例，可以使用 `help()` 在 Python 中查看：



In [ ]:
from omni.isaac.core import World
help(World)                 # 查看 World 类的方法与说明
help(World.step)            # 查看 step 方法的说明


同时，Isaac Lab 提供了丰富的环境与任务示例（位于 `isaaclab_tasks` 目录），包括机械臂操作、四足行走、灵巧手操作等，可直接作为二次开发的模板。



## 5.3.4 环境变量


常用的环境变量包括：

| 环境变量         | 作用                                                         |
| ---------------- | ------------------------------------------------------------ |
| `ISAAC_SIM_PATH` | Isaac Sim 的安装路径                                         |
| `OMNI_KIT_ARGS`  | 传递给 Omniverse Kit 的附加参数                              |
| `PYTHONPATH`     | 加入 `site-packages` 与扩展路径，便于导入 `omni.isaac.*` 模块 |
| `ISAACLAB_PATH`  | Isaac Lab 的安装路径                                         |

在 Windows 下，可通过设置环境变量让 Python 找到 Isaac 模块：



In [ ]:
set ISAAC_SIM_PATH=C:\Users\xxx\AppData\Local\ov\pkg\isaac_sim-2023.1.1
set PYTHONPATH=%ISAAC_SIM_PATH%\site-packages;%PYTHONPATH%


## 5.4.1 Isaac 的运用项目举例


**（1）与 ROS2 的应用**

Isaac Sim 与 ROS2 的桥接是机器人开发中最常见的用法之一。通过 `omni.isaac.ros2_bridge` 扩展，可以在仿真中发布相机图像、激光点云、里程计等话题，并订阅 ROS2 的控制指令，使真实的 ROS2 导航 / 规划 / 控制代码直接在仿真中运行，如图 5-4 所示：



![Isaac Sim 与 ROS2 的桥接](assets/fig5_6_ros2.png)

图 5-4 Isaac Sim 与 ROS2 的桥接示意

{: .figure-caption}

在启动脚本中加入桥接扩展即可启用：



In [ ]:
from omni.isaac.kit import SimulationApp

simulation_app = SimulationApp({
    "headless": False,
    "enable_camera_pose": True,
})
# 在命令行或扩展配置中加载 ros2_bridge 扩展
# --enable omni.isaac.ros2_bridge


**（2）Isaac Lab 强化学习训练**

Isaac Lab 是 Isaac 面向强化学习的开源框架，其训练流程如图 5-5 所示：创建任务环境 → 初始化仿真（USD + PhysX）→ 策略采样 → 执行动作 → 获取观测 / 奖励 → 更新策略，并利用 GPU 并行同时运行数千个环境。



![Isaac Lab 强化学习训练流程](assets/fig5_4_rl.png)

图 5-5 Isaac Lab 强化学习训练流程

{: .figure-caption}

一个典型的 Isaac Lab 环境构建代码如下：



In [ ]:
from isaaclab.envs import ManagerBasedEnv
from isaaclab_tasks.manager_based.manipulation.lift import mdp_cfg

# 环境配置（简化示例，实际使用需引用完整的任务配置）
env_cfg = mdp_cfg.LiftEnvCfg()
env = ManagerBasedEnv(cfg=env_cfg)

obs, _ = env.reset()
for i in range(200):
    action = env.action_space.sample()   # 随机动作（训练时由策略输出）
    obs, reward, terminated, truncated, info = env.step(action)
print("观测维度：", obs.shape)


ROS 的社区级概念是在 ROS 网络上进行代码发布的一种表现形式。

代码库的联合系统使得协作亦能被分发。这种从文件系统级别到社区一级的设计让不同开发者独立地发展和实施工作成为可能。正是因为这种分布式的结构，使得 ROS 迅速发展，软件仓库中包的数量以指数级增加。



***

